# Inspecting and Mitigating Bias in Machine Learning with Fairlearn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CDAC-lab/BUS3005-Resources/blob/main/Fairlearn_Bias_Inspection_and_Mitigation.ipynb)

Machine learning models are increasingly used to make decisions that affect people's lives &mdash; who gets a loan, who is shortlisted for a job, who is flagged for extra checks. When a model treats some groups of people systematically worse than others, we say the model is **biased** or **unfair**.

This tutorial is a gentle, hands-on introduction to **inspecting** and **mitigating** bias, using the open-source [**Fairlearn**](https://fairlearn.org/) toolkit. We will not go deep into the mathematics &mdash; the goal is to build intuition for three questions:

 * **Where** does bias show up in a model's predictions?
 * **How** can we *see* it clearly with the right visualisations?
 * **What** can we do to reduce it &mdash; and what does it cost us?

A key idea throughout: **a model can be biased even when we never give it the sensitive feature** (like sex or race). This surprises a lot of people, and we'll see exactly why it happens.

> This notebook adapts the Fairlearn [*GridSearch with Census Data*](https://fairlearn.org/main/auto_examples/plot_grid_search_census.html) example, with custom visualisations for navigating bias.


# The scenario: approving loans

We'll use the classic **UCI Adult Census** dataset. It contains data about ~48,000 people (age, education, working hours, occupation, and so on), and the original task is to predict whether a person earns more than \$50,000 a year.

To make the fairness question concrete, we'll **pretend this is a loan-approval problem**, exactly as the Fairlearn example does:

 * A label of **1** means *"this person repaid a loan in the past"*.
 * A label of **0** means *"this person did not repay"*.

We train a model to predict who is likely to repay, and imagine a bank uses that prediction to decide **who gets offered a loan**. If the model offers loans to one group far more often than another, that's a fairness problem worth understanding.

Below, click the **Play** icon beside each cell to run the code step by step as you work through the example.


First, we install and import the software libraries. This should only take a minute in Google Colab.

In [ ]:
#@title Install and import software libraries
!pip install fairlearn --quiet

# Data handling
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn import metrics as skm

# Fairlearn: fairness metrics and mitigation
from fairlearn.datasets import fetch_adult
from fairlearn.metrics import (
    MetricFrame,
    count,
    selection_rate,
    selection_rate_difference,
    demographic_parity_difference,
    plot_model_comparison,
)
from fairlearn.reductions import DemographicParity, GridSearch

pd.set_option("display.width", 120)
print("Libraries ready.")

We also define a few **custom visualisations** that we'll use to *navigate* bias. Don't worry about the plotting code &mdash; just run the cell. We'll explain each chart when we use it.

These charts are inspired by the Fairlearn dashboard and show:
1. **Disparity in predictions** &mdash; how often each group is offered a loan (the *selection rate*).
2. **Disparity in errors** &mdash; how the model's mistakes split into **underprediction** and **overprediction** for each group.


In [ ]:
#@title Define the bias-navigation visualisations
C_BLUE, C_ORANGE, C_GREY = "#1f77b4", "#ff7f0e", "#5a5a5a"

def group_bias_table(y_true, y_pred, sensitive):
    """Per-group selection rate, accuracy, under- and over-prediction rates."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    sensitive = np.asarray(sensitive)
    rows = {}
    for g in pd.unique(sensitive):
        m = sensitive == g
        yt, yp, n = y_true[m], y_pred[m], m.sum()
        rows[g] = {
            "count": int(n),
            "selection_rate": (yp == 1).mean(),
            "accuracy": (yp == yt).mean(),
            "underprediction": ((yp == 0) & (yt == 1)).sum() / n,  # false negatives / group
            "overprediction":  ((yp == 1) & (yt == 0)).sum() / n,  # false positives / group
        }
    return pd.DataFrame(rows).T

def _badge(ax, x, y, text, color):
    ax.text(x, y, text, fontsize=8, color="white", va="center", ha="center",
            bbox=dict(boxstyle="round,pad=0.25", fc=color, ec="none"))

def plot_selection_disparity(y_true, y_pred, sensitive, title="Disparity in predictions"):
    """Horizontal bars of selection rate per group, with Max/Min badges."""
    t = group_bias_table(y_true, y_pred, sensitive)
    overall = (np.asarray(y_pred) == 1).mean()
    disparity = t["selection_rate"].max() - t["selection_rate"].min()
    gmax, gmin = t["selection_rate"].idxmax(), t["selection_rate"].idxmin()
    groups = list(t.index); ypos = np.arange(len(groups))[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.9*len(t)+1.8))
    ax.barh(ypos, t["selection_rate"], color=C_BLUE, height=0.55, zorder=3)
    for yp, g in zip(ypos, groups):
        sr = t.loc[g, "selection_rate"]
        ax.text(sr+0.005, yp, f"{sr:.1%}", va="center", ha="left",
                fontsize=11, fontweight="bold", color=C_BLUE)
        ax.text(-0.02, yp, f"{g}", va="center", ha="right", fontsize=11)
        if g in (gmax, gmin):
            _badge(ax, -0.02, yp-0.32, "Max" if g == gmax else "Min", C_GREY)
    ax.set_yticks([]); ax.set_xlim(0, max(0.05, t["selection_rate"].max()*1.25))
    ax.set_xlabel("Selection rate  (fraction offered a loan)")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_title(f"{title}\nOverall selection rate: {overall:.1%}      "
                 f"Disparity (max - min): {disparity:.1%}",
                 loc="left", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()
    return t

def plot_error_disparity(y_true, y_pred, sensitive, title="Disparity in errors"):
    """Diverging bars: underprediction (orange, left) vs overprediction (blue, right)."""
    t = group_bias_table(y_true, y_pred, sensitive)
    overall_acc = (np.asarray(y_pred) == np.asarray(y_true)).mean()
    disparity = t["accuracy"].max() - t["accuracy"].min()
    gmax, gmin = t["accuracy"].idxmax(), t["accuracy"].idxmin()
    groups = list(t.index); ypos = np.arange(len(groups))[::-1]
    xmax = max(t["underprediction"].max(), t["overprediction"].max())*1.35 + 0.02
    fig, ax = plt.subplots(figsize=(10, 1.1*len(t)+2))
    for yp, g in zip(ypos, groups):
        under, over = t.loc[g, "underprediction"], t.loc[g, "overprediction"]
        ax.barh(yp, -under, color=C_ORANGE, height=0.55, zorder=3)
        ax.barh(yp,  over,  color=C_BLUE,   height=0.55, zorder=3)
        ax.text(-under-0.004, yp, f"{under:.1%}", va="center", ha="right",
                fontsize=10, color=C_ORANGE, fontweight="bold")
        ax.text( over+0.004, yp, f"{over:.1%}", va="center", ha="left",
                fontsize=10, color=C_BLUE, fontweight="bold")
        ax.text(-xmax, yp, f"{g}", va="center", ha="left", fontsize=11)
        ax.text(-xmax, yp-0.30, f"acc {t.loc[g,'accuracy']:.1%}", va="center",
                ha="left", fontsize=9, color=C_GREY)
        if g in (gmax, gmin):
            _badge(ax, -xmax+0.11, yp-0.30, "Max" if g == gmax else "Min", C_GREY)
    ax.axvline(0, color="black", lw=1)
    ax.set_yticks([]); ax.set_xlim(-xmax, xmax)
    ax.set_xlabel("<-  Underprediction (pred 0, true 1)        Overprediction (pred 1, true 0)  ->")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_title(f"{title}\nOverall accuracy: {overall_acc:.1%}      "
                 f"Disparity (max - min): {disparity:.1%}",
                 loc="left", fontsize=12, fontweight="bold")
    ax.legend(handles=[Patch(color=C_ORANGE, label="Underprediction  (denied, but would repay)"),
                       Patch(color=C_BLUE,   label="Overprediction  (approved, but would default)")],
              loc="lower right", fontsize=9, frameon=False)
    plt.tight_layout(); plt.show()
    return t

def plot_before_after(y_true, sensitive, preds, metric="selection_rate"):
    """Grouped bars comparing one metric across groups for several models."""
    tables = {name: group_bias_table(y_true, yp, sensitive) for name, yp in preds.items()}
    groups = list(next(iter(tables.values())).index)
    x = np.arange(len(groups)); w = 0.8/len(tables)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, (name, t) in enumerate(tables.items()):
        ax.bar(x+i*w, t[metric], w, label=name)
        for xi, v in zip(x+i*w, t[metric]):
            ax.text(xi, v+0.005, f"{v:.0%}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x+w*(len(tables)-1)/2); ax.set_xticklabels(groups)
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(f"{metric.replace('_',' ').title()} by group: before vs after mitigation")
    ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
    plt.tight_layout(); plt.show()

print("Visualisation functions ready.")

## Loading the data

We download the census data with Fairlearn's `fetch_adult` helper and print the first few rows. Each row is one person; the columns are the features we know about them.


In [ ]:
data = fetch_adult()
X_raw = data.data
# Our label: 1 if the person "repaid" (income > 50K in the original data), else 0
y = (data.target == ">50K").astype(int)

X_raw.head()

Notice that the data includes a **`sex`** column. This is what we'll call a **sensitive feature** &mdash; a characteristic that we care about being fair with respect to. Other common sensitive features are race, age, and disability status.

We'll now do something that seems like it *should* prevent unfairness: we **remove `sex` from the data the model learns from**, and keep it aside only so we can *measure* fairness later. Then we convert the remaining columns into numbers the model can use.


In [ ]:
# Pull the sensitive feature aside, and DROP it from the model's inputs
A = X_raw["sex"]                       # kept only for measuring fairness
X = X_raw.drop(labels=["sex"], axis=1) # the model never sees "sex"

# Turn text columns into numbers, and put everything on a common scale
X = pd.get_dummies(X)
X = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)

print(f"The model will be trained on {X.shape[1]} features - and 'sex' is NOT one of them.")

Next we split the data into a **training set** (to fit the model) and a **test set** (to evaluate it on people it has never seen).

In [ ]:
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, A, test_size=0.4, random_state=0, stratify=y
)

# Reset row numbers so everything lines up cleanly
for df in (X_train, X_test):
    df.reset_index(drop=True, inplace=True)
A_train = A_train.reset_index(drop=True)
A_test  = A_test.reset_index(drop=True)
y_train = np.asarray(y_train); y_test = np.asarray(y_test)

print(f"Training on {len(X_train):,} people, testing on {len(X_test):,} people.")

## Training a fairness-unaware model

We train an ordinary **logistic regression** classifier &mdash; a simple, widely used model. It knows nothing about fairness; it just tries to predict who will repay as accurately as possible. We call this the **unmitigated** model.


In [ ]:
unmitigated = LogisticRegression(solver="liblinear", fit_intercept=True)
unmitigated.fit(X_train, y_train)

y_pred_unmit = unmitigated.predict(X_test)
print("Model trained. Overall accuracy: "
      f"{skm.accuracy_score(y_test, y_pred_unmit):.1%}")

An accuracy in the mid-80s% sounds good! But **accuracy is an average over everyone**. It can hide the fact that the model works much better for some groups than others.

Let's use Fairlearn's `MetricFrame` to break performance down **by group**.


In [ ]:
metric_frame = MetricFrame(
    metrics={"accuracy": skm.accuracy_score,
             "selection_rate": selection_rate,
             "count": count},
    sensitive_features=A_test,
    y_true=y_test,
    y_pred=y_pred_unmit,
)

print("Overall:")
print(metric_frame.overall.to_string())
print("\nBy group:")
print(metric_frame.by_group.to_string())

Already we can see something concerning in the numbers. But numbers in a table are hard to *feel*. Let's visualise them.


### Visual 1: Disparity in predictions (who gets offered a loan?)

The **selection rate** is the fraction of a group that the model predicts as `1` &mdash; i.e. the fraction **offered a loan**. Under a fairness idea called **demographic parity**, we'd want every group to be offered loans at roughly the *same* rate.

The chart below shows the selection rate for each group. The **Max** and **Min** badges mark the most- and least-selected groups, and the title reports the overall rate and the **disparity** (the gap between the highest and lowest group).


In [ ]:
_ = plot_selection_disparity(y_test, y_pred_unmit, A_test)

Read this like the bar chart in a fairness dashboard: **the longer the bar, the more often that group is offered a loan.**

There is a large gap between the two bars. One group is offered loans at a much higher rate than the other &mdash; even though **the model was never told anyone's sex.** That gap is the *disparity in selection rate*, and it's the core fairness problem we want to reduce.


### Visual 2: Disparity in errors (underprediction vs overprediction)

Not all mistakes are the same. A model can be wrong in **two opposite ways**, and they harm different people:

 * **Underprediction** (predict `0` when the truth is `1`): the person *would* have repaid, but the model says *deny*. This is a **missed opportunity** &mdash; a harm to the applicant.
 * **Overprediction** (predict `1` when the truth is `0`): the person would *not* repay, but the model says *approve*. This is a **bad loan** &mdash; a harm to the lender (and sometimes to the borrower too).

The diverging chart below splits each group's errors into these two types: **orange bars point left for underprediction, blue bars point right for overprediction.** This is exactly the view in the second screenshot from the Fairlearn dashboard.


In [ ]:
_ = plot_error_disparity(y_test, y_pred_unmit, A_test)

This chart tells a richer story than accuracy alone. Look at **which group has the longer orange (underprediction) bar** &mdash; that group is being *denied loans they would have repaid* more often than the other. Two groups can even have similar-looking accuracy while their errors fall in completely different places, affecting people very differently.

**Key takeaway:** *"How accurate is the model?"* is the wrong question on its own. The better questions are *"Who does it make mistakes about, and what kind of mistakes?"*


### Why is the model biased if it never saw `sex`?

We deliberately deleted the `sex` column. So how can the model discriminate?

Because other features act as **proxies**. Things like occupation, hours worked per week, relationship status and capital gains are all **correlated with sex** in this data. The model can reconstruct the pattern from these stand-ins without ever seeing the label directly.

> **This is one of the most important lessons in fair ML: simply hiding a sensitive feature ("fairness through unawareness") usually does *not* make a model fair.** There are almost always enough correlated features left behind to recreate the bias.

So if deleting the feature doesn't work, what does? This is where **mitigation** comes in.


## Mitigating bias with GridSearch

Fairlearn's `GridSearch` takes our ordinary model and trains **many versions of it**, each nudged by a different amount toward the fairness goal. Some versions barely change (accurate but unfair); others push hard on fairness (fairer but less accurate). Together they map out the **trade-off** available to us.

We tell it:
 * the base model to use (logistic regression),
 * the fairness goal: **demographic parity** (equal selection rates across sex), and
 * how many versions to try (`grid_size`).

This trains many models, so it may take a minute or two in Colab.


In [ ]:
sweep = GridSearch(
    LogisticRegression(solver="liblinear", fit_intercept=True),
    constraints=DemographicParity(),
    grid_size=20,
)

sweep.fit(X_train, y_train, sensitive_features=A_train)
predictors = sweep.predictors_
print(f"Trained {len(predictors)} candidate models with different fairness/accuracy trade-offs.")

Many of these models are **dominated** &mdash; meaning another model is *both* more accurate *and* fairer, so there's no reason to pick them. We keep only the **non-dominated** models: the ones on the best-possible trade-off curve (the *Pareto front*).


In [ ]:
errors, disparities = [], []
for m in predictors:
    yp = m.predict(X_train)
    errors.append(1 - skm.accuracy_score(y_train, yp))
    disparities.append(
        demographic_parity_difference(y_train, yp, sensitive_features=A_train)
    )

all_results = pd.DataFrame(
    {"predictor": predictors, "error": errors, "disparity": disparities}
)

non_dominated = []
for row in all_results.itertuples():
    errors_for_lower_or_eq_disparity = all_results["error"][
        all_results["disparity"] <= row.disparity
    ]
    if row.error <= errors_for_lower_or_eq_disparity.min():
        non_dominated.append(row.predictor)

print(f"Kept {len(non_dominated)} non-dominated models (the best trade-offs).")

### Visual 3: The accuracy vs. fairness trade-off

Fairlearn's `plot_model_comparison` puts every model on one chart:
 * **horizontal axis = accuracy** (further right is better), and
 * **vertical axis = selection-rate difference** (closer to the bottom, i.e. `0`, is fairer).

The ideal model would sit in the **bottom-right corner**: perfectly accurate *and* perfectly fair. We can't reach it, but we can see how close we can get, and how the mitigated models compare to our original **unmitigated** one.


In [ ]:
predictions = {"unmitigated": y_pred_unmit}
for i, predictor in enumerate(non_dominated):
    predictions[f"mitigated_{i}"] = predictor.predict(X_test)

plot_model_comparison(
    x_axis_metric=skm.accuracy_score,
    y_axis_metric=selection_rate_difference,
    y_true=y_test,
    y_preds=predictions,
    sensitive_features=A_test,
    point_labels=True,
    show_plot=True,
)

Notice the shape: as we move the models toward fairness (downward), we usually give up only a *little* accuracy (a small move left). The disparity axis typically spans a much wider range than the accuracy axis &mdash; **we can remove most of the unfairness for a small accuracy cost.** That's the encouraging headline of this whole notebook.


## Selecting a better model

The chart shows the options; now we have to **choose one**. There's no universally "correct" pick &mdash; it depends on how a real organisation weighs fairness against accuracy.

Here we'll use a simple, defensible rule: **pick the fairest model that stays within 3 percentage points of the original model's accuracy.** In a real project you'd set this threshold with domain experts and affected communities.


In [ ]:
base_acc = skm.accuracy_score(y_test, y_pred_unmit)

candidates = []
for m in non_dominated:
    yp = m.predict(X_test)
    acc = skm.accuracy_score(y_test, yp)
    disp = selection_rate_difference(y_test, yp, sensitive_features=A_test)
    candidates.append((m, acc, disp))

# keep models within 3 percentage points of the original accuracy...
acceptable = [c for c in candidates if c[1] >= base_acc - 0.03] or candidates
# ...then choose the fairest of those (smallest selection-rate difference)
best_model, best_acc, best_disp = min(acceptable, key=lambda c: c[2])

y_pred_mit = best_model.predict(X_test)

print("                     accuracy   selection-rate disparity")
print(f"Unmitigated model :   {base_acc:6.1%}        "
      f"{selection_rate_difference(y_test, y_pred_unmit, sensitive_features=A_test):6.1%}")
print(f"Selected model    :   {best_acc:6.1%}        {best_disp:6.1%}")

We've traded a **small** amount of accuracy for a **large** reduction in disparity. Now let's confirm the improvement with the *same* visualisations we used before &mdash; this time on our chosen, bias-mitigated model.


### The mitigated model: disparity in predictions

In [ ]:
_ = plot_selection_disparity(y_test, y_pred_mit, A_test,
                             title="Disparity in predictions - MITIGATED model")

The two bars should now be **much closer in length** than before: the groups are offered loans at far more similar rates.

### The mitigated model: disparity in errors

In [ ]:
_ = plot_error_disparity(y_test, y_pred_mit, A_test,
                         title="Disparity in errors - MITIGATED model")

### Before vs. after, side by side

Finally, let's put the original and mitigated models next to each other so the change is unmistakable.


In [ ]:
plot_before_after(
    y_test, A_test,
    {"unmitigated": y_pred_unmit, "mitigated": y_pred_mit},
    metric="selection_rate",
)

The gap between the groups' bars shrinks dramatically after mitigation. We haven't made the model *perfect* &mdash; and we've accepted a small accuracy cost &mdash; but we've substantially reduced how differently it treats the two groups.


# Discussion points

Talk through these with a partner or in class:

 * Our rule was *"stay within 3 percentage points of the original accuracy."* Who should decide that threshold in a real bank? What happens if we make it stricter or looser?

 * We reduced **underprediction** and **overprediction** disparities, but they didn't vanish. Is "less unfair" good enough? When is it not?

 * We optimised for **demographic parity** (equal selection rates). But is *equal selection rate* always the right definition of fair? Imagine two groups that genuinely differ in repayment behaviour &mdash; does forcing equal rates help or hurt?

 * Deleting the `sex` column did **not** remove bias. What does this tell us about "colour-blind" or "we don't collect that data" approaches to fairness?

 * Who is harmed by an **underprediction** error versus an **overprediction** error? Should a fairness intervention treat those two harms equally?


# Important caveats

**Fairness is not a single number.** We used demographic parity, but there are many mathematical definitions of fairness (equal opportunity, equalised odds, and more), and it is provably **impossible to satisfy all of them at once**. Choosing a definition is an ethical and contextual decision, not just a technical one.

**Mitigation has a cost.** We traded accuracy for fairness. In some settings that trade is clearly worth it; in others it must be weighed carefully against real consequences.

**The tool is not the decision.** Fairlearn helps you *measure* and *reduce* disparity, but it cannot tell you what fairness *should mean* for your problem. That requires understanding the people affected, the harms at stake, and the law.

**Groups can hide sub-groups.** We only looked at sex. Real bias is often *intersectional* &mdash; e.g. affecting women of a particular age or background specifically. Inspecting one feature at a time can miss this.


# Further reading

 * [**Fairlearn documentation**](https://fairlearn.org/) &mdash; user guide, more examples, and API reference.
 * [**Fairlearn User Guide: Assessment**](https://fairlearn.org/main/user_guide/assessment/index.html) &mdash; a deeper tour of fairness metrics.
 * [**Fairlearn User Guide: Mitigation**](https://fairlearn.org/main/user_guide/mitigation/index.html) &mdash; other mitigation techniques beyond `GridSearch`.
 * Barocas, Hardt & Narayanan, [***Fairness and Machine Learning***](https://fairmlbook.org/) &mdash; a free, readable textbook on the whole topic.

# References

[1] Bird, S., Dudík, M., Edgar, R., et al. "Fairlearn: A toolkit for assessing and improving fairness in AI." *Microsoft Technical Report* MSR-TR-2020-32 (2020).

[2] Agarwal, A., Beygelzimer, A., Dudík, M., Langford, J., & Wallach, H. "A reductions approach to fair classification." *ICML* (2018).

---

*This notebook adapts the Fairlearn [GridSearch with Census Data](https://fairlearn.org/main/auto_examples/plot_grid_search_census.html) example, with custom bias-navigation visualisations inspired by the Fairlearn dashboard.*
